## Structured Output Streaming with Microsoft Agent Framework

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

### Create the Agent

In [ ]:
hr_agent = Agent(
    client=client,

    name="JobCandidateExtractor",

    instructions="""
    You are an HR assistant that extracts structured information about a job candidate from text.
    """
)

### Creating a Structured Output Schema using Pydantic

In [ ]:
from pydantic import BaseModel, ConfigDict

class CandidateProfile(BaseModel):
    """Structured job candidate profile"""
    name: str
    experience_years: int
    skills: list[str]
    current_role: str
    model_config = ConfigDict(extra="forbid")

### Run the Agent with Structured Output Streaming

In [ ]:
text_input = """
    Hi, I’m Alice Johnson. I've been a software engineer for 6 years,
    mainly working with Python, Azure, and React. Currently a senior developer at Contoso Ltd.
    """

response = await hr_agent.run(
    text_input, 
    options = {
        "response_format": CandidateProfile
    } 
)

print(response)

In [ ]:
if response.value:
    print("name:", response.value.name)
    print("experience_years:", response.value.experience_years)
    print("skills:", response.value.skills)
    print("current_role:", response.value.current_role)
else:
    print("Failed to parse response")